# Project 01 — Data collection and first profiling

## First-Time Buyer Affordability Pressure by Area

This notebook records the first data collection and profiling stage for the project. The source review establishes whether the official affordability workbook is suitable for analysis before findings are produced.

The workflow downloads the official ONS workbook into `data/raw/`, keeps the raw file separate from cleaned outputs, and performs first-pass profiling checks.


## Source collection

The raw workbook is collected from the ONS current dataset endpoint and stored in `data/raw/`. Raw files remain unchanged; cleaned outputs are saved separately.


In [1]:
from pathlib import Path
from urllib.parse import quote
from urllib.request import Request, urlopen
from datetime import date
import re
import shutil

import pandas as pd
import numpy as np


In [2]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
RAW_FILE = RAW_DIR / "aff1ratioofhousepricetoworkplacebasedearnings.xlsx"

RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

ONS_PATH = (
    "/peoplepopulationandcommunity/housing/datasets/"
    "ratioofhousepricetoworkplacebasedearningslowerquartileandmedian/"
    "current/aff1ratioofhousepricetoworkplacebasedearnings.xlsx"
)
SOURCE_URL = "https://www.ons.gov.uk/file?uri=" + quote(ONS_PATH, safe="")

def download_file(url, output_path):
    request = Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Accept": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,application/octet-stream,*/*",
        },
    )
    with urlopen(request) as response, open(output_path, "wb") as output_file:
        shutil.copyfileobj(response, output_file)

if RAW_FILE.exists():
    print("Raw workbook already exists. Download skipped.")
else:
    print("Downloading ONS workbook...")
    download_file(SOURCE_URL, RAW_FILE)
    print("Download complete.")

source_record = pd.DataFrame([{
    "source_name": "ONS House price to workplace-based earnings ratio",
    "source_url": SOURCE_URL,
    "local_file": str(RAW_FILE.relative_to(PROJECT_ROOT)),
    "download_checked_on": date.today().isoformat(),
    "file_size_mb": round(RAW_FILE.stat().st_size / (1024 * 1024), 2),
}])

source_record


Download complete.


,source_name,source_url,local_file,download_checked_on,file_size_mb
0,ONS House price to workplace-based earnings ratio,https://www.ons.gov.uk/file?uri=%2Fpeoplepopul...,data\raw\aff1ratioofhousepricetoworkplacebased...,2026-06-29,0.49


## Workbook inspection

ONS workbooks often include cover sheets, notes and metadata. The workbook structure is inspected before the data table is selected.


In [3]:
excel_file = pd.ExcelFile(RAW_FILE)

sheet_overview = pd.DataFrame({
    "sheet_number": range(1, len(excel_file.sheet_names) + 1),
    "sheet_name": excel_file.sheet_names,
})

sheet_overview


,sheet_number,sheet_name
0,1,Contents
1,2,Metadata
2,3,Terms and Conditions
3,4,1a
4,5,1b
5,6,1c
6,7,2a
7,8,2b
8,9,2c
9,10,3a


In [4]:
for sheet_name in excel_file.sheet_names:
    print("\n" + "=" * 80)
    print(f"Sheet: {sheet_name}")
    print("=" * 80)
    preview = pd.read_excel(RAW_FILE, sheet_name=sheet_name, header=None, nrows=12)
    display(preview)



Sheet: Contents


,0,1
0,Contents,NaN
1,Metadata,Information on the ratio of house price to wor...
2,Terms and Conditions,Terms and conditions
3,Country and region - Median,NaN
4,Table 1a (Median house price),"Median house price by country and region, Engl..."
5,Table 1b (Median earnings),Median gross annual workplace-based earnings b...
6,Table 1c (Median affordability ratio),Ratio of median house price to median gross an...
7,Country and region - Lower quartile,NaN
8,Table 2a (Lower quartile house price),Lower quartile house price by country and regi...
9,Table 2b (Lower quartile earnings),Lower quartile gross annual workplace-based ea...



Sheet: Metadata


,0
0,Metadata
1,Affordability ratios calculated by dividing ho...
2,For detailed information on aspects of the qua...
3,Housing affordability in England and Wales QMI
4,Earnings
5,Earnings data are taken from the Annual Survey...
6,• full-time earnings;\n• annual earnings where...
7,Annual estimates of earnings are based on the...
8,For detailed information on aspects of the qua...
9,"Annual Survey of Hours and Earnings, Low pay a..."



Sheet: Terms and Conditions


...

,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,32
0,Table 6a - Lower quartile house price by local...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Country/Region code,Country/Region name,Local authority code,Local authority name,Year ending Sep 1997,Year ending Sep 1998,Year ending Sep 1999,Year ending Sep 2000,Year ending Sep 2001,Year ending Sep 2002,...,Year ending Sep 2016,Year ending Sep 2017,Year ending Sep 2018,Year ending Sep 2019,Year ending Sep 2020,Year ending Sep 2021,Year ending Sep 2022,Year ending Sep 2023,Year ending Sep 2024,Year ending Sep 2025
2,E12000001,North East,E06000001,Hartlepool,25500,27000,26000,27500,25950,27708,...,70000,75625,77500,83875,83250,85000,85000,93000,92953,100000
3,E12000001,North East,E06000002,Middlesbrough,28000,27000,28500,27000,23000,25000,...,83625,79375,83000,85000,87000,90000,93000,93000,100000,105000
4,E12000001,North East,E06000003,Redcar and Cleveland,32000,34000,33000,35500,36750,37000,...,92000,90000,93875,91000,96500,105000,105000,112000,116000,124225
5,E12000001,North East,E06000004,Stockton-on-Tees,36500,36500,37000,38050,39000,41500,...,98995,105000,105000,106375,110000,119984,116750,125000,125000,128000
6,E12000001,North East,E06000005,Darlington,32000,32500,32500,35000,36000,36950,...,87500,95000,94500,95000,102000,112000,110000,115000,120000,127500
7,E12000001,North East,E06000047,County Durham,29000,29000,30000,31000,31000,32000,...,69000,69950,72000,75000,72000,81000,84000,85000,85000,92500
8,E12000001,North East,E06000057,Northumberland,34950,35000,37000,38000,38500,40000,...,105000,107500,110000,114000,115000,130000,124995,135000,138500,139988
9,E12000001,North East,E08000021,Newcastle upon Tyne,36000,36500,38500,39000,40000,46000,...,112000,114995,115249,119000,115000,133000,135000,140000,140000,150000



Sheet: 6b


,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,32
0,Table 6b - Lower quartile gross annual workpla...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Country/Region code,Country/Region name,Local authority code,Local authority name,1997,1998,1999,2000,2001,2002,...,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024.0,2025.0
2,E12000001,North East,E06000001,Hartlepool,11354,10180,10835,10507,11655,11598,...,20386.0,19470.0,18506.0,19238.0,19798.0,21084.0,20916.0,22081.0,24505.0,26716.0
3,E12000001,North East,E06000002,Middlesbrough,11182,11395,11649,11954,12380,12894,...,19032.0,19026.0,19076.0,20806.0,20367.0,21173.0,22989.0,23879.0,25884.0,27262.0
4,E12000001,North East,E06000003,Redcar and Cleveland,12606,12152,14000,14942,14048,14463,...,16684.0,17676.0,18340.0,19126.0,19445.0,19376.0,19662.0,23182.0,24547.0,27486.0
5,E12000001,North East,E06000004,Stockton-on-Tees,11896,11213,11622,12736,13904,12896,...,19180.0,19199.0,19178.0,21060.0,22381.0,22187.0,23635.0,24522.0,25378.0,27342.0
6,E12000001,North East,E06000005,Darlington,10206,10237,11743,11800,12104,12916,...,20244.0,18779.0,20170.0,20180.0,19319.0,20616.0,21773.0,22919.0,25108.0,27445.0
7,E12000001,North East,E06000047,County Durham,[x],[x],[x],[x],[x],[x],...,18029.0,18635.0,19532.0,19845.0,19869.0,20006.0,21879.0,22943.0,25154.0,27003.0
8,E12000001,North East,E06000057,Northumberland,[x],[x],[x],[x],[x],[x],...,17999.0,18918.0,19819.0,20085.0,19988.0,21424.0,22906.0,24051.0,25074.0,27330.0
9,E12000001,North East,E08000021,Newcastle upon Tyne,11656,11808,12272,13853,13672,13896,...,19367.0,20875.0,20034.0,20698.0,21484.0,22732.0,23758.0,26071.0,27203.0,29211.0



Sheet: 6c


,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
0,Table 6c - Ratio of lower quartile house price...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Country/Region code,Country/Region name,Local authority code,Local authority name,1997,1998,1999,2000,2001,2002,...,2017.00,2018.00,2019.00,2020.00,2021.00,2022.00,2023.00,2024.00,2025.00,5-Year Average
2,E12000001,North East,E06000001,Hartlepool,2.25,2.65,2.4,2.62,2.23,2.39,...,3.88,4.19,4.36,4.20,4.03,4.06,4.21,3.79,3.74,3.97
3,E12000001,North East,E06000002,Middlesbrough,2.5,2.37,2.45,2.26,1.86,1.94,...,4.17,4.35,4.09,4.27,4.25,4.05,3.89,3.86,3.85,3.98
4,E12000001,North East,E06000003,Redcar and Cleveland,2.54,2.8,2.36,2.38,2.62,2.56,...,5.09,5.12,4.76,4.96,5.42,5.34,4.83,4.73,4.52,4.97
5,E12000001,North East,E06000004,Stockton-on-Tees,3.07,3.26,3.18,2.99,2.8,3.22,...,5.47,5.48,5.05,4.91,5.41,4.94,5.10,4.93,4.68,5.01
6,E12000001,North East,E06000005,Darlington,3.14,3.17,2.77,2.97,2.97,2.86,...,5.06,4.69,4.71,5.28,5.43,5.05,5.02,4.78,4.65,4.99
7,E12000001,North East,E06000047,County Durham,[x],[x],[x],[x],[x],[x],...,3.75,3.69,3.78,3.62,4.05,3.84,3.70,3.38,3.43,3.68
8,E12000001,North East,E06000057,Northumberland,[x],[x],[x],[x],[x],[x],...,5.68,5.55,5.68,5.75,6.07,5.46,5.61,5.52,5.12,5.56
9,E12000001,North East,E08000021,Newcastle upon Tyne,3.09,3.09,3.14,2.82,2.93,3.31,...,5.51,5.75,5.75,5.35,5.85,5.68,5.37,5.15,5.14,5.44


## Candidate data table detection

The scan below searches the top of each sheet for terms likely to appear in the affordability table. It acts as a profiling aid rather than a substitute for analytical judgement.


In [5]:
def normalise_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().lower()

search_terms = ["lower quartile", "median", "earnings", "ratio", "area", "local authority"]
sheet_matches = []
header_candidates = []

for sheet_name in excel_file.sheet_names:
    preview = pd.read_excel(RAW_FILE, sheet_name=sheet_name, header=None, nrows=40, dtype=str)
    sheet_text = " ".join(preview.fillna("").astype(str).to_numpy().ravel()).lower()
    matched_terms = [term for term in search_terms if term in sheet_text]
    sheet_matches.append({
        "sheet_name": sheet_name,
        "matched_terms": ", ".join(matched_terms),
        "match_count": len(matched_terms),
    })

    for row_number, row in preview.iterrows():
        row_text = " | ".join(normalise_text(value) for value in row.tolist() if normalise_text(value))
        score = sum(term in row_text for term in ["area", "code", "year", "median", "lower", "quartile", "earnings", "ratio"])
        if score >= 3:
            header_candidates.append({
                "sheet_name": sheet_name,
                "zero_based_row": row_number,
                "excel_row": row_number + 1,
                "score": score,
                "row_text": row_text[:300],
            })

sheet_match_summary = pd.DataFrame(sheet_matches).sort_values(["match_count", "sheet_name"], ascending=[False, True])
header_candidates_df = pd.DataFrame(header_candidates).sort_values(["score", "sheet_name", "excel_row"], ascending=[False, True, True])

display(sheet_match_summary)
display(header_candidates_df.head(20))


,sheet_name,matched_terms,match_count
1,Metadata,"lower quartile, median, earnings, ratio, area,...",6
0,Contents,"lower quartile, median, earnings, ratio, local...",5
17,5c,"median, earnings, ratio, local authority",4
20,6c,"lower quartile, earnings, ratio, local authority",4
5,1c,"median, earnings, ratio",3
8,2c,"lower quartile, earnings, ratio",3
11,3c,"median, earnings, ratio",3
14,4c,"lower quartile, earnings, ratio",3
16,5b,"median, earnings, local authority",3
19,6b,"lower quartile, earnings, local authority",3


,sheet_name,zero_based_row,excel_row,score,row_text
12,Metadata,1,2,7,affordability ratios calculated by dividing ho...
18,2c,0,1,4,table 2c - ratio of lower quartile house price...
22,4c,0,1,4,table 4c - ratio of lower quartile house price...
26,6c,0,1,4,table 6c - ratio of lower quartile house price...
3,Contents,10,11,4,table 2c (lower quartile affordability ratio) ...
7,Contents,18,19,4,table 4c (lower quartile affordability ratio) ...
11,Contents,26,27,4,table 6c (lower quartile affordability ratio) ...
14,Metadata,15,16,4,by dividing the house price for a given area b...
15,1c,0,1,3,table 1c - ratio of median house price to medi...
16,2a,0,1,3,table 2a - lower quartile house price by count...


## Initial data profile

The highest-scoring candidate is loaded as a structured first pass. The selected sheet and header row remain visible for review before the cleaning stage.


In [6]:
if not header_candidates_df.empty:
    best_candidate = header_candidates_df.iloc[0]
    DATA_SHEET = best_candidate["sheet_name"]
    HEADER_ROW = int(best_candidate["zero_based_row"])
else:
    DATA_SHEET = excel_file.sheet_names[0]
    HEADER_ROW = 0

print(f"Selected sheet: {DATA_SHEET}")
print(f"Selected header row: {HEADER_ROW} zero-based, Excel row {HEADER_ROW + 1}")

raw_df = pd.read_excel(RAW_FILE, sheet_name=DATA_SHEET, header=HEADER_ROW)
print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]:,}")
raw_df.head()


Selected sheet: Metadata
Selected header row: 1 zero-based, Excel row 2
Rows: 24
Columns: 1


,"Affordability ratios calculated by dividing house prices by gross annual earnings, based on the median and lower quartiles of both house prices and earnings. The earnings data are from the Annual Survey of Hours and Earnings which provides a snapshot of earnings at April in each year. Earnings relate to gross full-time individual earnings on a place of work basis. The house price statistics come from the House Price Statistics for Small Areas, which report the median and lower quartile price paid for residential property and refer to a 12 month period with April in the middle (year ending September). Statistics are available at country, region, county and local authority district level in England and Wales."
0,For detailed information on aspects of the qua...
1,Housing affordability in England and Wales QMI
2,Earnings
3,Earnings data are taken from the Annual Survey...
4,• full-time earnings;\n• annual earnings where...


In [7]:
column_profile = pd.DataFrame({
    "column_name": raw_df.columns.astype(str),
    "dtype": [str(dtype) for dtype in raw_df.dtypes],
    "non_null_count": raw_df.notna().sum().values,
    "missing_count": raw_df.isna().sum().values,
    "missing_pct": (raw_df.isna().mean().values * 100).round(2),
})

duplicate_row_count = raw_df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_row_count:,}")
display(column_profile)


Exact duplicate rows: 1


,column_name,dtype,non_null_count,missing_count,missing_pct
0,Affordability ratios calculated by dividing ho...,object,24,0,0.0


In [8]:
profile_output = CLEANED_DIR / "initial_column_profile.csv"
source_record_output = RAW_DIR / "source_record.csv"

column_profile.to_csv(profile_output, index=False)
source_record.to_csv(source_record_output, index=False)

print(f"Saved column profile to: {profile_output.relative_to(PROJECT_ROOT)}")
print(f"Saved source record to: {source_record_output.relative_to(PROJECT_ROOT)}")


Saved column profile to: data\cleaned\initial_column_profile.csv
Saved source record to: data\raw\source_record.csv


## Profiling summary and next analytical step

The next notebook will clean and reshape the data after confirming the correct data sheet, area fields, year field, lower-quartile measures and row granularity.

Unusual values remain part of the review at this stage. The most unaffordable areas may look like outliers, but they are likely to be analytically important rather than errors.
